# Validation and Edge Cases

This notebook demonstrates the package behavior for ambiguous or incomplete sampled landscapes. It focuses on three cases that matter when preparing real micromagnetic transition networks:

1. duplicate saddles are rejected during `LandscapeGraph` validation;
2. equal-energy saddles produce deterministic branch ordering;
3. disconnected sampled landscapes can still be visualized, but the root is marked as synthetic.

The examples are intentionally small so the logic is easy to inspect before plugging in Merrill.jl-style LEM/NEB outputs.

In [1]:
using Pkg
example_dir = basename(pwd()) == "examples" ? pwd() : joinpath(pwd(), "examples")
if !isfile(joinpath(example_dir, "Project.toml"))
    example_dir = pwd()
end
Pkg.activate(example_dir)
Pkg.develop(path=abspath(joinpath(example_dir, "..")))
Pkg.instantiate()

using DisconnectivityGraphs
using PlotlyJS

capture_error(f) = try
    f()
    nothing
catch err
    sprint(showerror, err)
end

struct NotebookPlot{P}
    plot::P
end

function html_escape(value::AbstractString)
    return replace(value,
        "&" => "&amp;",
        "\"" => "&quot;",
        "<" => "&lt;",
        ">" => "&gt;")
end

Base.show(io::IO, ::MIME"text/plain", ::NotebookPlot) = print(io, "Plotly interactive figure")
Base.show(io::IO, mime::MIME"application/vnd.plotly.v1+json", value::NotebookPlot) = show(io, mime, value.plot)
function Base.show(io::IO, mime::MIME"text/html", value::NotebookPlot)
    html = sprint(show, mime, value.plot)
    print(io, """
    <iframe srcdoc="$(html_escape(html))"
            style="width:100%; height:680px; border:0;"
            sandbox="allow-scripts allow-same-origin">
    </iframe>
    """)
end

show_plotly(plot) = NotebookPlot(plot)

function plot_disconnectivity_tree(tree; title, order=leaf_order(tree))
    layout = tree_layout(tree; order=order)
    xs = Float64[]
    ys = Float64[]
    for segment in layout.segments
        append!(xs, [segment.x0, segment.x1, NaN])
        append!(ys, [Float64(segment.y0), Float64(segment.y1), NaN])
    end

    leaf_ids = [node_id for node_id in eachindex(tree.nodes) if isempty(tree.nodes[node_id].children)]
    internal_ids = [node_id for node_id in eachindex(tree.nodes) if !isempty(tree.nodes[node_id].children)]

    leaf_trace = scatter(
        x=[layout.x[node_id] for node_id in leaf_ids],
        y=[Float64(layout.y[node_id]) for node_id in leaf_ids],
        mode="markers+text",
        text=[String(only(tree.nodes[node_id].minima)) for node_id in leaf_ids],
        textposition="bottom center",
        marker=attr(size=12, color="#2563eb", line=attr(color="white", width=1)),
        hovertext=["minimum $(only(tree.nodes[node_id].minima))<br>E=$(round(Float64(tree.nodes[node_id].energy); digits=3))" for node_id in leaf_ids],
        hoverinfo="text",
        showlegend=false,
    )

    internal_trace = scatter(
        x=[layout.x[node_id] for node_id in internal_ids],
        y=[Float64(layout.y[node_id]) for node_id in internal_ids],
        mode="markers+text",
        text=[tree.nodes[node_id].synthetic ? "synthetic root" : "merge" for node_id in internal_ids],
        textposition="top center",
        marker=attr(
            size=[tree.nodes[node_id].synthetic ? 15 : 11 for node_id in internal_ids],
            color=[tree.nodes[node_id].synthetic ? "#dc2626" : "#64748b" for node_id in internal_ids],
            symbol=[tree.nodes[node_id].synthetic ? "diamond" : "circle" for node_id in internal_ids],
            line=attr(color="white", width=1)
        ),
        hovertext=[tree.nodes[node_id].synthetic ? "synthetic root<br>E=$(round(Float64(tree.nodes[node_id].energy); digits=3))<br>minima=$(join(string.(tree.nodes[node_id].minima), ", "))" : "merge<br>E=$(round(Float64(tree.nodes[node_id].energy); digits=3))<br>minima=$(join(string.(tree.nodes[node_id].minima), ", "))" for node_id in internal_ids],
        hoverinfo="text",
        showlegend=false,
    )

    branch_trace = scatter(
        x=xs,
        y=ys,
        mode="lines",
        line=attr(color="#334155", width=2.5),
        hoverinfo="skip",
        showlegend=false,
    )

    show_plotly(Plot([branch_trace, internal_trace, leaf_trace], Layout(
        title=title,
        xaxis=attr(visible=false),
        yaxis=attr(title="energy"),
        width=820,
        height=460,
        showlegend=false,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=attr(l=60, r=30, t=70, b=55),
    )))
end

component_summary(partition) = join(["{" * join(string.(component), ", ") * "}" for component in partition], " | ")

function plot_component_counts(landscape; thresholds=collect(range(0.0, 2.2; length=12)), title="Connected components by threshold")
    counts = [length(component_partition(landscape, threshold)) for threshold in thresholds]
    hover = ["threshold=$(round(Float64(threshold); digits=2))<br>components=$(count)<br>$(component_summary(component_partition(landscape, threshold)))" for (threshold, count) in zip(thresholds, counts)]
    show_plotly(Plot([scatter(
        x=Float64.(thresholds),
        y=counts,
        mode="lines+markers",
        line=attr(color="#0f766e", width=3),
        marker=attr(size=9, color="#14b8a6"),
        hovertext=hover,
        hoverinfo="text",
        showlegend=false,
    )], Layout(
        title=title,
        xaxis=attr(title="threshold energy"),
        yaxis=attr(title="number of connected components", dtick=1),
        width=820,
        height=420,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=attr(l=60, r=30, t=70, b=55),
    )))
end


  Activating project at `~/Github/DisconnectivityGraphs.jl/examples`
   Resolving package versions...
  No Changes to `~/Github/DisconnectivityGraphs.jl/examples/Project.toml`
  No Changes to `~/Github/DisconnectivityGraphs.jl/examples/Manifest.toml`


plot_component_counts (generic function with 1 method)

## Duplicate Saddle Validation

A disconnectivity graph needs one representative saddle per undirected minimum pair. If the same pair is supplied twice, the package now fails early instead of silently choosing one and producing a misleading branch structure.

In [2]:
duplicate_message = capture_error() do
    LandscapeGraph(
        [Minimum(:A, 0.0), Minimum(:B, 1.0)],
        [Saddle(:A, :B, 2.0), Saddle(:B, :A, 2.0)],
    )
end

println(duplicate_message)

duplicate saddle between minima A and B; keep only one representative saddle per undirected pair


## Deterministic Equal-Energy Merges

Equal-energy saddles are common in simplified or symmetry-related examples. The tree should not flip left-to-right just because the saddle list was given in a different order.

In [3]:
minima = [
    Minimum(:A, 0.0),
    Minimum(:B, 1.0),
    Minimum(:C, 2.0),
]

saddles_forward = [
    Saddle(:C, :A, 4.0),
    Saddle(:B, :C, 4.0),
    Saddle(:A, :B, 4.0),
]

tree_forward = disconnectivity_tree(LandscapeGraph(minima, saddles_forward))
tree_reverse = disconnectivity_tree(LandscapeGraph(minima, reverse(saddles_forward)))

(leaf_order(tree_forward), leaf_order(tree_reverse), tree_forward.nodes[tree_forward.root].energy)

([:A, :B, :C], [:A, :B, :C], 4.0)

The leaf order stays the same even when the input saddle order is reversed. That is useful for regression testing, manuscript figures, and comparing landscape snapshots across field or temperature changes.

### Visual Check: Forward Saddle Order

This plot shows the deterministic tree built from the original equal-energy saddle ordering.


In [4]:
plot_disconnectivity_tree(tree_forward; title="Equal-energy merges: forward saddle order")


Plotly interactive figure

### Visual Check: Reversed Saddle Order

Reversing the saddle list yields the same leaf order and branch geometry, which is the stability property we want for repeated analyses.


In [5]:
plot_disconnectivity_tree(tree_reverse; title="Equal-energy merges: reversed saddle order")


Plotly interactive figure

## Disconnected Sampled Landscapes

Incomplete NEB sampling is normal early in a workflow. In that situation the package can still return a plottable tree, but the final root is synthetic rather than a real saddle. That distinction matters physically: it means the sampled network is incomplete, not that a single transition state has been located.

In [6]:
disconnected = LandscapeGraph(
    [Minimum(:plus_x, 0.0), Minimum(:minus_x, 0.1), Minimum(:vortex, 0.4)],
    [Saddle(:plus_x, :minus_x, 1.8)],
)

tree = disconnectivity_tree(disconnected)

(
    has_synthetic_root(tree),
    tree.nodes[tree.root].synthetic,
    tree.nodes[tree.root].energy,
    component_partition(disconnected, 2.0),
)

(true, true, 1.8, [[:plus_x, :minus_x], [:vortex]])

### Visual Check: Synthetic Root Highlight

The disconnected sampled landscape remains plottable, but the synthetic root is highlighted in red so it cannot be mistaken for a real transition state.


In [7]:
plot_disconnectivity_tree(tree; title="Disconnected sampled landscape with synthetic root")


Plotly interactive figure

### Threshold Sweep

This threshold scan shows how many connected components remain as the allowed saddle energy rises. Because the `:vortex` state has no sampled connecting saddle, one component remains isolated throughout the sweep.


In [8]:
plot_component_counts(disconnected; thresholds=collect(range(0.0, 2.2; length=12)), title="Disconnected landscape: components vs threshold")


Plotly interactive figure

## Micromagnetic Interpretation

For micromagnetic modeling, these checks are practical rather than cosmetic:

- duplicate saddles often indicate a preprocessing bug or conflicting NEB summaries;
- deterministic ties keep repeated analyses visually stable when barriers are symmetry-related;
- synthetic roots warn that the current network is still missing transitions, so the graph should not yet be over-interpreted as a complete reversal topology.